In [1]:
##background:
#I assume that the prior year loss run is carried forward from last analysis, rather than being provided again this year. If it were provided again this year, I would compare to an existing file from last year.
#I run this exercise after the triangle exercise #2. This is relevant only because I take the output file from that exercise as the input file for this exercise.
#I wanted to try out fuzzyjoin since I think it has better performance on weirder joins compared to dplyr

##user inputs:
#source data: go to the notebook cell headed with "##create worbook for output, test saving a copy"
#selects: go to the notebook cell headed with "##make selects so that they can easily be read in the notebook"; specify input file, input sheets, and output file
#packages: if you don't have them, install the packages under "## packages, run once globally"

##TODO: Study what can cause some of these seeming error such as falling paid or disappearing claims. There could be a good reason.
##TODO: Store long-term history.
##TODO: Try to have pattern recognition on distribution of losses to look for anomalies. Probably not appropriate for this small dataset but would make sense for a larger one.
##TODO: plots/markdown?
##TODO: make stylechange calls dynamic; right now, I'm just relying on fixed column positions for date columns
##TODO: make autowidth work better; right now, I'm just relying on fixed width because auto is too weak and then occasionally I have overly wide title cells

##Approach: I think the more standard approach is to use dataframes for everything and then look at them in pivoted representation. I want to try keeping things in matrix form.

In [2]:
## packages, run once globally
# install.packages("openxlsx2")
# install.packages("lubridate")
# install.packages("tidyverse")
# install.packages("zoo")
# install.packages("fuzzyjoin")

In [3]:
##create worbook for output, test saving a copy
library(openxlsx2)
input_path = "output/PYRQ_output.xlsx"
output_path = "output/PYRQ_output.xlsx"
input_sheet_current = "ex1_curr"
input_sheet_prior = "ex1_prior"

wb <- wb_load(input_path)
wb_save(wb,output_path,overwrite = TRUE)

#create styles
dec3="0.000"
dec0="#,##0"
decY="###0"
dt="YYYY/MM/DD"

#function to create a sheet destructively; can use this to clear a sheet also
newsheet_destructive <- function(name) {
    if (name %in% wb$sheet_names) wb$remove_worksheet(sheet=name)
    wb$add_worksheet(name, tabColour = "pink")
}

#function to populate a sheet
popsheet <- function(name,frame,title,discussion,style,titlerow,startcolumn) {
    wb$add_data(sheet = name, title, start_row = titlerow, start_col = startcolumn, row_names=FALSE, na="")
    wb$add_data(sheet = name, discussion, start_row = titlerow+1, start_col = startcolumn, row_names=FALSE, na="")
    wb$add_data(sheet = name, frame, start_row = titlerow+4, start_col = startcolumn, row_names=TRUE, na="")
    wb$add_numfmt(sheet=name, numfmt=style, 
         dims=wb_dims(rows=(titlerow+4):(nrow(frame)+(titlerow+4)), cols=(startcolumn+1):(ncol(frame)+startcolumn+1)))
    wb$add_font(sheet=name, bold=TRUE,
        dims=wb_dims(rows=(titlerow+4):(nrow(frame)+(titlerow+4)),cols=startcolumn:startcolumn))
    wb$add_font(sheet=name, bold=TRUE,
        dims=wb_dims(rows=(titlerow):(titlerow),cols=startcolumn:(ncol(frame)+startcolumn+1)))
    wb$add_font(sheet=name, bold=TRUE,
        dims=wb_dims(rows=(titlerow+4):(titlerow+4),cols=startcolumn:(ncol(frame)+startcolumn+1)))
}

#function to get max row and column
bounds <- function(name) {
  ws <- wb$worksheets[[which(wb$sheet_names == name)]]
  cc <- ws$sheet_data$cc
  max_row <- max(as.integer(cc$row_r))
  max_col <- max(col2int(cc$c_r))
  return(c(max_row, max_col))
}

#function to change column style
stylechange <- function(name,style,row,lb,ub) {
    wb$add_numfmt(sheet=name, numfmt=style, 
         dims=wb_dims(rows=1:row,cols=lb:ub))
}

#function to set column widths
autowidth <- function(name,col=-1,w=-1) {
    if (col==-1) ncols=bounds(name)[2] else ncols=col
    if (w == -1) width="auto" else width=w
    wb$set_col_widths(name, cols=1:ncols, widths=width)
}

In [4]:
##look at list of worksheets
sheets <- wb_get_sheet_names(wb)
data.frame(index = seq_along(sheets), name = sheets)


,index,name
,<int>,<chr>
ex1_curr,1,ex1_curr
ex1_prior,2,ex1_prior
ex2_tri,3,ex2_tri
initial diagnostics,4,initial diagnostics
triangles,5,triangles
notes,6,notes


In [5]:
##read the triangle, create matrix, create variables to be used for checks

library(lubridate)
library(tidyverse)
library(fuzzyjoin)

#I don't want scientific notation in output
options(scipen=999)

#load the loss runs
c <- wb_to_df(wb, sheet=input_sheet_current)
c = cbind(c,source=matrix(input_sheet_current,nrow(c),1))
p <- wb_to_df(wb, sheet=input_sheet_prior)
p = cbind(p,source=matrix(input_sheet_prior,nrow(p),1))

#create appended version to make some checks quicker
s = rbind(c,p)
rownames(s) <- NULL

#create joined version to make some checks quicker
j = fuzzy_full_join(c,p, by=c("claim_num"="claim_num"), match_fun=list(`==`))
rownames(j) <- NULL
colnames(j) <- sub("\\.x", ".c", colnames(j))
colnames(j) <- sub("\\.y", ".p", colnames(j))

#function to elementwise compare matrix elements without freaking out over NA; this version will count NA=NaN
equal <- function(x, y) {(is.na(x) == is.na(y)) & (is.na(x) | x==y)}

#ancient date that will be less than any relevant date
ancient <- ISOdate(1900, 1, 1, tz="UTC")
future <- ISOdate(3000, 1, 1, tz="UTC")

#frame for errors
test_errors <- data.frame(short=character(),message=character(), error=numeric())


Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr   1.2.1     ✔ readr   2.2.0
✔ forcats 1.0.1     ✔ stringr 1.6.0
✔ ggplot2 4.0.3     ✔ tibble  3.3.1
✔ purrr   1.2.2     ✔ tidyr   1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [6]:
##tests on the claim numbers

#build the claim numbers we expect based on the A0001-type pattern for current
c = cbind(c,claim_alpha=substr(c[,'claim_num'],1,1))
c = cbind(c,claim_suffix=substr(c[,'claim_num'],2,nchar(c[,'claim_num'])))
c = cbind(c,claim_suffix_numeric=as.numeric(c[,'claim_suffix']))
c_claim_suffix_numeric_min = min(c[,'claim_suffix_numeric'])
c_claim_suffix_numeric_max = max(c[,'claim_suffix_numeric'])
c_claim_suffix_numeric_expected = (c_claim_suffix_numeric_min:c_claim_suffix_numeric_max)
c_cne = data.frame(claim_num=paste0('A',formatC(c_claim_suffix_numeric_expected,width=4,flag="0")))
c_cne = cbind(c_cne,source=matrix(input_sheet_current,nrow(c_cne),1))

#build the claim numbers we expect based on the A0001-type pattern for prior
p = cbind(p,claim_alpha=substr(p[,'claim_num'],1,1))
p = cbind(p,claim_suffix=substr(p[,'claim_num'],2,nchar(p[,'claim_num'])))
p = cbind(p,claim_suffix_numeric=as.numeric(p[,'claim_suffix']))
p_claim_suffix_numeric_min = min(p[,'claim_suffix_numeric'])
p_claim_suffix_numeric_max = max(p[,'claim_suffix_numeric'])
p_claim_suffix_numeric_expected = (p_claim_suffix_numeric_min:p_claim_suffix_numeric_max)
p_cne = data.frame(claim_num=paste0('A',formatC(p_claim_suffix_numeric_expected,width=4,flag="0")))
p_cne = cbind(p_cne,source=matrix(input_sheet_prior,nrow(p_cne),1))

#stack the two
s_cne = rbind(c_cne,p_cne)

#look for claims that we expect
T <- fuzzy_left_join(s_cne, s, by=c("claim_num"="claim_num","source"="source"), match_fun=list(`==`,`==`))
T <- T[is.na(T[["claim_num.y"]]),]
T <- T[c("claim_num.x","source.x")]
T

#create error messages
short = "maybe missing claims"
message = "claim numbers which potentially should show up but don't"
discussion = "could be non-sequential claim numbers, CWP/dormant claims removed, portfolio/segmentation change, error correction, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

,claim_num.x,source.x
,<chr>,<chr>
10,A0010,ex1_curr


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1


In [7]:
##test for duplicate claim numbers
T <- s %>% group_by(claim_num,source) %>% summarize (recordcount=n())
T <- T[T[["recordcount"]]>=2,]
T

#create error messages
short = "claim dupes"
message = "claim numbers which show up multiple times"
discussion = "could be multiple features on the same claim, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by claim_num and source.
ℹ Output is grouped by claim_num.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(claim_num, source))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


claim_num,source,recordcount
<chr>,<chr>,<int>


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0


In [8]:
##close flag vs case reserve
T <- cbind(s,zero_case = (s[["case_loss"]]==0))
T <- T[T[["zero_case"]]!=(T[["claim_status"]]=="Closed"),]
T

#create error messages
short = "closure flag vs case"
message = "closure flag which doesn't seem to match case reserves"
discussion = "could be something about claims process, timing issue, closure being defined at a different level of aggregation than the claim, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num", "source","claim_status","case_loss","clsd_date"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

,claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,source,zero_case
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>
10,A0011,2021-05-14,2022-10-19,2022-12-02,Closed,62968,61968,1000,ex1_curr,FALSE


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1


In [9]:
##close flag vs close date
T <- cbind(s,close_date_exists = (coalesce(s[["clsd_date"]],ancient)>ancient))
T <- T[T[["close_date_exists"]]!=(T[["claim_status"]]=="Closed"),]
T

#create error messages
short = "closure flag vs close date"
message = "closure flag which doesn't seem to match close date"
discussion = "could be something about claims process, timing issue, reopen conventions, closure being defined at different levels of aggregation, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num", "source","claim_status","case_loss","clsd_date"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,source,close_date_exists
<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0


In [10]:
##report date vs close date
T <- s[coalesce(s[["clsd_date"]],ancient)>ancient,]
T <- T[T[["rep_date"]]>T[["clsd_date"]],]
T

#create error messages
short = "close date < report date"
message = "close date < report date"
discussion = "could be reopen convention, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num", "source","rep_date","clsd_date"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,source
<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0


In [11]:
##loss date vs report date
T <- s[coalesce(s[["rep_date"]],ancient)>ancient,]
T <- T[T[["acc_date"]]>T[["rep_date"]],]
T

#create error messages
short = "report date < loss date"
message = "report date < loss date"
discussion = "not sure what could cause this"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num", "source","rep_date","acc_date"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,source
<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0


In [12]:
##dates missing
T <- s[coalesce(s[["rep_date"]],ancient)==ancient | coalesce(s[["acc_date"]],ancient)==ancient,]
T

#create error messages
short = "missing loss or report date"
message = "missing loss or report date"
discussion = "not sure what could cause this"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num", "source","rep_date","acc_date"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,source
<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0


In [13]:
##negative paid or case
T <- s[s[["pd_loss"]]<0 | s[["case_loss"]]<0,]
T

#create error messages
short = "negative paid or case"
message = "negative paid or case"
discussion = "could be recoveries, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num", "source","pd_loss","case_loss"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,source
<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0


In [14]:
##paid+case<>reported
T <- s[s[["pd_loss"]]+s[["case_loss"]]!=s[["rep_loss"]],]
T

#create error messages
short = "paid+case vs rptd"
message = "paid+case vs rptd"
discussion = "could be recoveries, expenses, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num", "source","pd_loss","case_loss","rep_loss"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

,claim_num,acc_date,rep_date,clsd_date,claim_status,rep_loss,pd_loss,case_loss,source
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
3,A0003,2019-08-27,2020-10-05,NA,Open,86564,78926,8638,ex1_curr


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0
paid+case vs rptd,paid+case vs rptd,1


In [15]:
##paid decreased
T <- j[j[["pd_loss.c"]]<j[["pd_loss.p"]],]
T <- T[!is.na(T[["pd_loss.c"]]) & !is.na(T[["pd_loss.p"]]),]
T

colnames(T) <- sub("\\.c", ".current", colnames(T))
colnames(T) <- sub("\\.p", ".prior", colnames(T))

#create error messages
short = "paid decreased"
message = "paid decreased"
discussion = "could be recoveries, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors
colnames(T)

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    stylechange(short,dt,nrow(T)+7,12,14)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num.current", "source.current","claim_num.prior", "source.prior","pd_loss.current","pd_loss.prior"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

,claim_num.c,acc_date.c,rep_date.c,clsd_date.c,claim_status.c,rep_loss.c,pd_loss.c,case_loss.c,source.c,claim_num.p,acc_date.p,rep_date.p,clsd_date.p,claim_status.p,rep_loss.p,pd_loss.p,case_loss.p,source.p
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
1,A0001,2019-02-03,2019-06-01,2021-01-21,Closed,72568,72568,0,ex1_curr,A0001,2019-02-03,2019-06-01,2021-01-21,Closed,74570,74570,0,ex1_prior
8,A0008,2021-01-07,2021-02-18,NA,Open,78300,23792,54508,ex1_curr,A0008,2021-01-07,2021-02-18,NA,Open,78300,25792,52508,ex1_prior


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0
paid+case vs rptd,paid+case vs rptd,1


[1] "claim_num.current"    "acc_date.current"     "rep_date.current"    
 [4] "clsd_date.current"    "claim_status.current" "rep_loss.current"    
 [7] "pd_loss.current"      "case_loss.current"    "source.current"      
[10] "claim_num.prior"      "acc_date.prior"       "rep_date.prior"      
[13] "clsd_date.prior"      "claim_status.prior"   "rep_loss.prior"      
[16] "pd_loss.prior"        "case_loss.prior"      "source.prior"

In [16]:
##date decreased or became blank
#I'm not looking at close date since maybe this will become blank upon reopen

T <- j[!is.na(j[["claim_num.c"]]) & !is.na(j[["claim_num.p"]]),]
T <- T[coalesce(T[["acc_date.c"]],ancient)<coalesce(T[["acc_date.p"]],ancient) | coalesce(T[["rep_date.c"]],ancient)<coalesce(T[["rep_date.p"]],ancient),]
T

colnames(T) <- sub("\\.c", ".current", colnames(T))
colnames(T) <- sub("\\.p", ".prior", colnames(T))

#create error messages
short = "date decreased"
message = "loss/report date decreased or disappeared"
discussion = "could be revisions, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    stylechange(short,dt,nrow(T)+7,12,14)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num.current", "source.current","claim_num.prior", "source.prior","acc_date.current","acc_date.prior","rep_date.current","rep_date.prior"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

claim_num.c,acc_date.c,rep_date.c,clsd_date.c,claim_status.c,rep_loss.c,pd_loss.c,case_loss.c,source.c,claim_num.p,acc_date.p,rep_date.p,clsd_date.p,claim_status.p,rep_loss.p,pd_loss.p,case_loss.p,source.p
<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0
paid+case vs rptd,paid+case vs rptd,1


In [17]:
##dates changed
T <- j[j[["acc_date.c"]]!=j[["acc_date.p"]] | j[["rep_date.c"]]!=j[["rep_date.p"]],]
T <- T[!is.na(T[["acc_date.c"]]) & !is.na(T[["acc_date.p"]]),]
T

colnames(T) <- sub("\\.c", ".current", colnames(T))
colnames(T) <- sub("\\.p", ".prior", colnames(T))

#create error messages
short = "loss or report date changed"
message = "loss or report date changed"
discussion = "could be error correction, reopen, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    stylechange(short,dt,nrow(T)+7,12,14)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num.current", "source.current","claim_num.prior", "source.prior","acc_date.current","acc_date.prior","rep_date.current","rep_date.prior"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

,claim_num.c,acc_date.c,rep_date.c,clsd_date.c,claim_status.c,rep_loss.c,pd_loss.c,case_loss.c,source.c,claim_num.p,acc_date.p,rep_date.p,clsd_date.p,claim_status.p,rep_loss.p,pd_loss.p,case_loss.p,source.p
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
7,A0007,2020-08-21,2022-07-03,2022-07-15,Closed,7843,7843,0,ex1_curr,A0007,2020-08-11,2022-07-03,2022-07-15,Closed,7843,7843,0,ex1_prior


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0
paid+case vs rptd,paid+case vs rptd,1


In [18]:
##claim disappeared
T <- j[is.na(j[["claim_num.c"]]) & !is.na(j[["claim_num.p"]]),]
T

colnames(T) <- sub("\\.c", ".current", colnames(T))
colnames(T) <- sub("\\.p", ".prior", colnames(T))

#create error messages
short = "claim number disappeared"
message = "claim number present in prior but not current"
discussion = "could be non-sequential claim numbers, CWP/dormant claims removed, portfolio/segmentation change, error correction, rolling time window, something else"
error = nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    stylechange(short,dt,nrow(T)+7,12,14)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num.current", "source.current","claim_num.prior", "source.prior"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

,claim_num.c,acc_date.c,rep_date.c,clsd_date.c,claim_status.c,rep_loss.c,pd_loss.c,case_loss.c,source.c,claim_num.p,acc_date.p,rep_date.p,clsd_date.p,claim_status.p,rep_loss.p,pd_loss.p,case_loss.p,source.p
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
13,NA,NA,NA,NA,NA,NA,NA,NA,NA,A0010,2021-04-26,2021-12-15,NA,Open,17378,12512,4866,ex1_prior


short,message,error
<chr>,<chr>,<int>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0
paid+case vs rptd,paid+case vs rptd,1


In [19]:
##did we get new claims since last time?
T <- j[!is.na(j[["claim_num.c"]]) & is.na(j[["claim_num.p"]]),]
T

#create error messages
short = "did we get new claims since last time?"
message = "1 if we got no new claims since prior loss run, else 0"
discussion = "could be that we truly didn't have any"
if (nrow(T)==0) error = 1 else error = 0

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

,claim_num.c,acc_date.c,rep_date.c,clsd_date.c,claim_status.c,rep_loss.c,pd_loss.c,case_loss.c,source.c,claim_num.p,acc_date.p,rep_date.p,clsd_date.p,claim_status.p,rep_loss.p,pd_loss.p,case_loss.p,source.p
,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<date>,<date>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
10,A0011,2021-05-14,2022-10-19,2022-12-02,Closed,62968,61968,1000,ex1_curr,NA,NA,NA,NA,NA,NA,NA,NA,NA
11,A0012,2021-04-16,2021-06-07,2022-03-21,Closed,44128,44128,0,ex1_curr,NA,NA,NA,NA,NA,NA,NA,NA,NA
12,A0013,2022-10-27,2023-03-26,NA,Open,8399,6586,1813,ex1_curr,NA,NA,NA,NA,NA,NA,NA,NA,NA


short,message,error
<chr>,<chr>,<dbl>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0
paid+case vs rptd,paid+case vs rptd,1


In [20]:


##new claims with a date < max date of prior loss run

#I don't know the evaluation date of the prior loss run, but I'm guessing that it's <= the max date in the prior loss run
priorEval_LB = pmax(max(p[,c("acc_date")]),max(p[,c("rep_date")]),max(coalesce(p[,c("clsd_date")],ancient)))
priorEval_LB

T <- j[!is.na(j[["claim_num.c"]]) & is.na(j[["claim_num.p"]]),]
T <- T[T[["rep_date.c"]]<priorEval_LB | coalesce(T[["clsd_date.c"]],future)<priorEval_LB,]
T <- T[c("claim_num.c", "acc_date.c","rep_date.c","clsd_date.c","source.c")]
T

colnames(T) <- sub("\\.c", ".current", colnames(T))
colnames(T) <- sub("\\.p", ".prior", colnames(T))

#create error messages
short = "retroactive changes"
message = "new claims added in current loss run, with report/close date likely prior to the evaluation date of the prior loss run"
discussion = "could be timing issues, something else"
error=nrow(T)

#add errors to table
test_errors=rbind(test_errors,data.frame(short=short,message=message,error=error))
test_errors

#add errors to worksheet
if (error>0) {
    newsheet_destructive(short)
    popsheet(short,T,short,discussion,dec0,1,1)
    stylechange(short,dt,nrow(T)+7,3,5)
    wb$add_fill(sheet=short, dims = (wb_dims(rows=1:(nrow(T)+7),cols=1+match(c("claim_num.current","acc_date.current","rep_date.current","clsd_date.current", "source.current"), colnames(T)))), color = wb_color(hex = "F8C8DC"))
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}

[1] "2022-07-15"

,claim_num.c,acc_date.c,rep_date.c,clsd_date.c,source.c
,<chr>,<date>,<date>,<date>,<chr>
11,A0012,2021-04-16,2021-06-07,2022-03-21,ex1_curr


short,message,error
<chr>,<chr>,<dbl>
maybe missing claims,claim numbers which potentially should show up but don't,1
claim dupes,claim numbers which show up multiple times,0
closure flag vs case,closure flag which doesn't seem to match case reserves,1
closure flag vs close date,closure flag which doesn't seem to match close date,0
close date < report date,close date < report date,0
report date < loss date,report date < loss date,0
missing loss or report date,missing loss or report date,0
negative paid or case,negative paid or case,0
paid+case vs rptd,paid+case vs rptd,1


In [21]:
##summarize errors

#create error messages
short = "error summary"
discussion = "table of error types and counts"

#add errors to worksheet
{
    newsheet_destructive(short)
    popsheet(short,test_errors,short,discussion,dec0,1,1)
    autowidth(short,,12)
    wb_save(wb, output_path, overwrite = TRUE)
}